# Kaggle / Colab runner: stress-detection-dreaddit

**Kaggle settings (right panel):** Accelerator → *GPU T4 x1* (or x2), Internet → *On* (needs a phone-verified account).
**Secrets (Add-ons → Secrets):** `HF_TOKEN` = your HuggingFace read token (after accepting the MentalBERT terms).

Run the cells top to bottom, then run **only your** work-package cell. At the end, download `outputs.zip`
from the Output panel, unzip it into your local clone, and commit `results/` + `figures/` on your branch.

In [ ]:
REPO_URL = "https://github.com/Iamdevsuyash/stress-detection-dreaddit.git"
BRANCH = "main"
!git clone -q -b $BRANCH $REPO_URL repo
%cd repo

In [ ]:
!pip install -q lime tabulate bitsandbytes
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
# MentalBERT ships pytorch_model.bin -> needs torch >= 2.6. If older, run: !pip install -q -U "torch>=2.6"

In [ ]:
import os
try:                                    # Kaggle
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:                                # Colab
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: gated models (MentalBERT/MentalRoBERTa, Llama) will fail to download.")

In [ ]:
!python -m src.data --stats

## Work packages: run only yours

In [ ]:
# Suyash: LSTM baselines (downloads GloVe, ~820 MB)
!mkdir -p data/glove && wget -q https://nlp.stanford.edu/data/glove.6B.zip && unzip -o -j -q glove.6B.zip glove.6B.100d.txt -d data/glove && rm glove.6B.zip
!bash scripts/run_baselines.sh

In [ ]:
# Saurabh: encoders x 3 seeds. Set DAPT=1 to also run the domain-adaptive pretraining stretch goal.
!MODELS="bert-base mentalbert roberta-base mentalroberta" DAPT=0 bash scripts/run_transformers.sh

In [ ]:
# Suyash: LLM zero-/few-shot
!LLM="Qwen/Qwen2.5-7B-Instruct" bash scripts/run_llm.sh

In [ ]:
# Sarthak: explainability + error analysis (needs models/bert-base__seed13 and models/mentalbert__seed13
# from the Saurabh cell in THIS session; otherwise re-create them here. --no_record keeps Saurabh's results untouched)
!python -m src.train_transformer --model bert-base --seed 13 --save --no_record
!python -m src.train_transformer --model mentalbert --seed 13 --save --no_record
!bash scripts/run_explain.sh

## Package outputs

In [ ]:
!python -m src.eval summarize
!zip -q -r /kaggle/working/outputs.zip results figures 2>/dev/null || zip -q -r outputs.zip results figures
print("Download outputs.zip from the Output panel (Kaggle) or the Files pane (Colab).")